# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-5.4-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [2]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

프롬프트: 프랑스의 수도는
생성: 프랑스의 수도는 **파리**입니다.

프롬프트: Q: 프랑스의 수도는 어디인가요? A:
생성: 프랑스의 수도는 **파리**입니다.

프롬프트: 프랑스의 수도 도시는
생성: 프랑스의 수도는 **파리**입니다.



---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [ ]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero)) 

=== Zero-shot ===
- "연출은 좋았지만 2시간이 너무 길었다" → **부정**
- "최고의 영화, 또 보고 싶다" → **긍정**


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → **중립에 가까운 혼합 감성** | **5/10**  
입력: "최고의 영화, 또 보고 싶다" → **긍정** | **10/10**


In [5]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

나쁜

설명:  
- **행복한 : 슬픈**은 반대말 관계입니다.  
- 따라서 **좋은 : 나쁜**도 반대말 관계가 되어야 합니다.


In [6]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

Few-shot 프롬프트:
다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :

생성된 완성:
좋은 : **나쁜**


---
## 2. 역할(Role) 설정 

In [7]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
대부분의 **건강한 성인**이라면 **하루 커피 4잔 정도는 보통 괜찮은 편**입니다.  
다만 “괜찮다”는 건 **카페인 총량**과 **개인 상태**에 따라 달라져요.

### 보통 기준
- 성인 카페인 권장 상한: **하루 400mg 이하**
- 커피 1잔(아메리카노 기준)은 대략 **카페인 80~150mg** 정도로 차이가 큼
- 그래서 **4잔이면 대략 320~600mg**까지 갈 수 있어,  
  **커피 종류/크기**에 따라 상한을 넘을 수도 있습니다.

### 4잔이 특히 조심해야 하는 경우
- **임신 중**: 보통 **하루 200mg 이하** 권장
- **불면증, 불안, 두근거림**이 있는 경우
- **고혈압, 부정맥, 위염/역류성 식도염**이 있는 경우
- 카페인에 **민감한 체질**
- 오후 늦게 마셔서 **수면에 영향**이 있는 경우

### 이런 증상이 있으면 줄이는 게 좋아요
- 잠이 잘 안 옴
- 심장이 두근거림
- 손 떨림
- 속쓰림/위 불편
- 불안감 증가

### 현실적인 팁
- **오후 2~3시 이후는 줄이기**
- 진한 커피 대신 **연한 커피/디카페인 섞기**
- 커피 외에 **에너지음료, 차, 초콜릿**의 카페인도 합산하기

원하시면 제가 **“당신이 마시는 커피 종류 기준으로 4잔이 괜찮은지”** 대략 계산해드릴게요.




In [8]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

=== 영양사 역할 ===
대부분의 **건강한 성인**이라면 하루 **커피 4잔**은 **가능한 범위**인 경우가 많습니다. 다만 **커피 종류, 잔 크기, 개인의 카페인 민감도**에 따라 달라집니다.

### 카페인 양 대략
- **아메리카노 1잔(약 240ml)**: 보통 **80~150mg**
- **드립커피 1잔**: **95~200mg**
- **에스프레소 1샷**: **약 60~75mg**

그래서 **커피 4잔 = 대략 320~800mg** 정도까지도 될 수 있어요.  
일반적으로 성인 권장 상한은 **하루 카페인 400mg 이하**로 보는 경우가 많습니다.

### 괜찮을 수 있는 경우
- 커피를 마셔도 **심장 두근거림, 불안, 손떨림, 속쓰림, 불면**이 없을 때
- **오전~이른 오후**에만 마실 때
- 다른 카페인 음료(에너지드링크, 진한 차, 초콜릿 등)를 많이 안 먹을 때

### 조심해야 하는 경우
- **임신 중**: 보통 **하루 200mg 이하** 권장
- **불면증, 불안장애, 위염/역류성 식도염, 부정맥, 고혈압이 잘 조절 안 됨**
- 카페인에 예민해서 **1~2잔만 마셔도 잠이 안 오는 경우**
- **저녁 늦게** 마시는 습관

### 실용적인 팁
- 4잔을 마시더라도 **작은 잔으로 나눠서**
- **오후 2~3시 이후는 줄이기**
- 공복 커피는 속을 자극할 수 있어 **식후에 마시기**
- 커피에 **설탕, 시럽, 크림**이 많으면 칼로리가 크게 늘어납니다

원하시면 제가 **평소 드시는 커피 종류(아메리카노/라떼/믹스/캡슐 등)** 기준으로 **4잔의 카페인 mg를 더 정확히 계산**해드릴게요.


In [9]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 인공지능(AI)은 **사람처럼 생각하고 학습하며 문제를 해결하도록 만든 컴퓨터 기술**입니다.  
쉽게 말해, 컴퓨터가 단순히 정해진 명령만 따르는 것이 아니라 **데이터를 바탕으로 패턴을 배우고, 예측하고, 판단하는 것처럼 동작**하게 만드는 기술입니다.

예를 들면:

- **음성 인식**: 휴대폰이 내 말을 알아듣는 기능
- **추천 시스템**: 유튜브나 넷플릭스가 취향에 맞는 콘텐츠를 추천하는 기능
- **챗봇**: 고객 문의에 자동으로 답하는 서비스
- **이미지 인식**: 사진 속 사람이나 사물을 구분하는 기능

인공지능은 크게 두 가지로 이해할 수 있습니다.

1. **좁은 인공지능(약한 AI)**  
   - 특정 작업만 잘하는 AI입니다.
   - 현재 우리가 사용하는 대부분의 AI가 여기에 해당합니다.

2. **범용 인공지능(강한 AI)**  
   - 사람처럼 다양한 일을 스스로 이해하고 수행하는 AI입니다.
   - 아직은 완전히 실현되지 않았습니다.

인공지능의 장점은 **빠른 처리, 반복 작업 자동화, 정확도 향상** 등이 있고,  
단점은 **데이터 편향, 오작동 가능성, 개인정보 문제** 등이 있습니다.

원하시면 제가 이어서 **“인공지능의 작동 원리”**나 **“일상생활 속 인공지능 예시”**도 쉽게 설명드릴게요.

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(AI, Artificial Intelligence)은 **사람이 수행하던 지능적 작업을 컴퓨터가 하도록 만드는 기술**입니다.  
즉, 기계가 **학습하고, 판단하고, 예측하고, 문제를 해결**할 수 있게 하는 분야입니다.

### 핵심 개념
- **학습**: 데이터를 통해 규칙이나 패턴을 익힘
- **추론**: 학습한 내용을 바탕으로 결론을 도출함
- **판단**: 여러 선택지 중 적절한 결과를 선택함
- **자동화**: 반복적이거나 복잡한 작업을 사람 대신 수행함

### 예시


---
## 3. 출력 형식 지정 

In [10]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

=== Markdown ===
- **장점 1: 출퇴근 시간이 줄어든다**  
  이동 시간이 없어져 더 효율적으로 시간을 활용할 수 있습니다.

- **장점 2: 업무 집중도가 높아질 수 있다**  
  조용한 환경을 만들면 방해 요소가 줄어들어 집중하기 좋습니다.

- **장점 3: 일과 생활의 균형을 맞추기 쉽다**  
  집에서 일하므로 개인 일정과 업무를 더 유연하게 조정할 수 있습니다.

- **단점 1: 소통이 불편할 수 있다**  
  대면 대화가 줄어들어 협업이나 빠른 의사소통이 어려울 수 있습니다.

- **단점 2: 업무와 휴식의 경계가 흐려질 수 있다**  
  집에서 일하다 보면 퇴근 후에도 업무가 이어지기 쉽습니다.

- **단점 3: 고립감을 느낄 수 있다**  
  동료와 직접 만날 기회가 적어 외로움이나 소속감 저하를 느낄 수 있습니다.




In [11]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목 | 장점 | 단점 |
|---|---|---|
| 시간 활용 | 출퇴근 시간이 없어 효율적이다 | 업무와 개인 시간의 경계가 흐려질 수 있다 |
| 업무 환경 | 자신에게 맞는 환경에서 일할 수 있다 | 집 환경에 따라 집중이 어려울 수 있다 |
| 비용 절감 | 교통비·식비 등 지출을 줄일 수 있다 | 전기·인터넷 등 개인 부담이 늘 수 있다 |


---
### 출력 형식 (표 / JSON)

In [12]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

=== Table ===
| 요인 | 설명 | 대응 |
|---|---|---|
| 식각 균일도 | 웨이퍼 내/웨이퍼 간 식각 속도 차이로 CD 편차, 잔사, 과식각이 발생해 수율이 저하됨 | 장비 튜닝, 가스 유량/압력 최적화, 챔버 상태 관리, 균일도 모니터링 |
| 선택비(Selectivity) | 목표막 대비 마스크/하부막 식각이 과도하면 패턴 손상, 마스크 소모, 하부층 손상이 발생함 | 식각 화학종 최적화, 엔드포인트 제어, 마스크 두께/재질 최적화 |
| 잔사 및 파티클 | 식각 부산물, 폴리머, 파티클이 패턴 막힘이나 결함을 유발해 오픈/쇼트 불량을 초래함 | 챔버 클리닝, 필터 관리, 레시피 최적화, 세정 공정 강화 |


In [13]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

=== JSON ===
{"factors":[{"name":"식각 균일도(Uniformity)","action":"웨이퍼 전면의 식각 속도와 깊이를 균일하게 유지하도록 가스 유량, 압력, RF 파워, 온도 조건을 최적화한다."},{"name":"선택비(Selectivity)","action":"목표층은 충분히 식각하면서 마스크나 하부층 손상을 최소화하도록 식각 화학과 공정 조건을 조정한다."},{"name":"과식각/언더식각(Over/Under Etch) 제어","action":"엔드포인트 검출과 타이밍 제어를 통해 잔류막을 제거하되 불필요한 과식각으로 인한 CD 손실과 손상을 방지한다."}]}

파싱 성공: ['factors']


---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [14]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

=== CoT 없음 ===
아니요, **추가 배치가 필요하지 않습니다.**

계산해보면:

- 전체 인원: 25명
- A팀 인원: 10명
- A팀 비율: 10 ÷ 25 = **0.4 = 40%**

조건이 **“40% 이상”**이므로,  
A팀이 정확히 40%인 현재도 조건을 **충족**합니다.

따라서 **추가 인원 배치는 필요 없습니다.**


In [15]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

=== CoT 적용 ===
단계별로 계산해 보겠습니다.

1. **전체 인원 수 확인**
   - A팀: 10명
   - B팀: 15명
   - C팀: 5명  
   - **총합 = 10 + 15 + 5 = 30명**

2. **A팀 비율 계산**
   - A팀 비율 = 10 / 30 = 0.333... = **33.3%**

3. **기준과 비교**
   - 기준: **40% 이상**
   - 현재 A팀 비율: **33.3%**

4. **판단**
   - 33.3%는 40% 이상이 아니므로,
   - **A팀에 추가 인원 배치는 필요하지 않습니다.**

**결론: 지금 추가 배치가 필요하지 않습니다.**


In [16]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

연필 5자루의 값은 5 × 500 = 2500원이고,  
지우개 3개의 값은 3 × 300 = 900원입니다.

따라서 총 금액은 2500 + 900 = **3400원**입니다.

답: **3400원**


In [17]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

연필 값과 지우개 값을 각각 계산해보겠습니다.

1. 연필 5자루: 5 × 500원 = 2,500원  
2. 지우개 3개: 3 × 300원 = 900원  

합하면,

2,500원 + 900원 = **3,400원**

답: **3400원**


In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

---
## 5. System + User 프롬프트 

In [ ]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

In [ ]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

---
## 6. Self-check Prompting

In [ ]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)

---
## 7. Constraint Prompting

In [ ]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)